# Traditional Methods - Code

In this notebook, we will use the KL-Sum and LexRank extractive algorithms to create summaries from our dataset.

In [72]:
# Install necessary libraries

!pip install -q rouge_score bert_score sumy lexrank textblob

In [73]:
# Import necessary libraries

import pandas as pd
import numpy as np
import json
from textblob import TextBlob
from scipy.spatial.distance import jensenshannon
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize, word_tokenize
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import KFold
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from lexrank import LexRank
from lexrank.mappings.stopwords import STOPWORDS
import textwrap

import warnings
warnings.filterwarnings("ignore")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/joerussell/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [74]:
# Load the dataset

train_df = pd.read_json('../data/amasum_train_processed.json', orient='records', lines=True)
test_df = pd.read_json('../data/amasum_test_processed.json', orient='records', lines=True)

print("Train dataset shape: ", train_df.shape)
print("Test dataset shape: ", test_df.shape)

Train dataset shape:  (1243, 8)
Test dataset shape:  (308, 8)


In [75]:
# Create dictionary of gold summaries for evaluation

gold_summaries = test_df[['verdict', 'pros', 'cons']].to_dict(orient='records')

## Summarization

For each product, our gold summaries include an overall verdict, a list of pros, and a list of cons. We will generate these three summaries separately. We can apply our extractive models directly to the review text to generate our verdicts. However, since these methods are unsupervised, we cannot generate pros and cons in the same way. Instead, we will divide sentences in the lists of reviews by sentiment. Our extractive models will be applied to positive sentences to generate a list of pros and to negative sentences to generate a list of cons.

In [76]:
# Function to get a list of positive sentences from reviews

def get_positive_sentences(reviews):
    all_sentences = [sent for review in reviews for sent in sent_tokenize(review)]
    positive_sentences = [
        sent for sent in all_sentences if TextBlob(sent).sentiment.polarity > 0.2
    ]
    return positive_sentences

In [77]:
# Function to get a list of negative sentences from reviews

def get_negative_sentences(reviews):
    all_sentences = [sent for review in reviews for sent in sent_tokenize(review)]
    negative_sentences = [
        sent for sent in all_sentences if TextBlob(sent).sentiment.polarity < -0.2
    ]
    return negative_sentences

### KL-Sum

The KL-Sum algorithm selects sentences based on the similarity of their word frequency distribution to that of all reviews for that product. Thus, we will create separate functions to compute the word distribution of a piece of text and to calculate the KL divergence between two distributions.

```{admonition} Note
:class: info
Since KL-Sum is a fast and relatively simple algorithm, all steps are implemented below to further demonstrate how it works. The [sumy](https://pypi.org/project/sumy/) Python package has a very effective KLSummarizer implementation.
```

In [78]:
# Compute word distribution for a given text

def compute_distribution(text):
  tokens = word_tokenize(text)
  counts = Counter(tokens)
  total = sum(counts.values())
  distribution = {word: count / total for word, count in counts.items()}
  return distribution

:::{dropdown} Code Explanation
```python
def compute_distribution(text):
  tokens = word_tokenize(text)
```
The input text is tokenized into individual words using `word_tokenize`. This prepares the text for frequency analysis at the word level.

```python
  counts = Counter(tokens)
  total = sum(counts.values())
```
A `Counter` counts the number of times each token appears. The total number of tokens is calculated to normalize the frequency values into probabilities.

```python
  distribution = {word: count / total for word, count in counts.items()}
  return distribution
```
Each word's frequency is converted into a probability by dividing its count by the total number of tokens. This yields a normalized probability distribution over the words in the input text, which is essential for KL divergence calculations in KL-Sum.
:::


In [79]:
# Calculate KL divergence between two word distributions

def kl_div(pd, ps):
  kl_sum = 0
  for word, pd_prob in pd.items():
    ps_prob = ps.get(word, 1e-10)
    kl_sum += pd_prob * np.log(pd_prob / ps_prob)
  return kl_sum

:::{dropdown} Code Explanation
```python
def kl_div(pd, ps):
  kl_sum = 0
```
This function computes the Kullback-Leibler (KL) divergence between two probability distributions `pd` (the document distribution) and `ps` (the summary distribution). `kl_sum` accumulates the divergence score.

```python
  for word, pd_prob in pd.items():
    ps_prob = ps.get(word, 1e-10)
```
For each word in the document distribution, retrieve its probability. Then, attempt to fetch the corresponding probability from the summary distribution. If the word is not present, use a very small default probability (`1e-10`) to avoid division by zero or log of zero.

```python
    kl_sum += pd_prob * np.log(pd_prob / ps_prob)
  return kl_sum
```
The KL divergence formula is applied: $ KL(P || Q) = \sum p(x) \log\left(\frac{p(x)}{q(x)}\right) $. This measures how much information is lost when the summary distribution `ps` is used to approximate the full document distribution `pd`.
:::


As mentioned above, we will generate the verdict, pros, and cons separately with three different functions for clarity.

In [80]:
def kl_sum_verdict(reviews, summary_length=2):

    all_reviews = " ".join(reviews)
    sentences = sent_tokenize(all_reviews)
    pd = compute_distribution(all_reviews)

    summary = []

    while len(summary) < summary_length:
        closest_sentence = None
        min_kl = np.inf

        for sentence in sentences:
            if sentence in summary:
                continue

            potential_summary = " ".join(summary + [sentence])
            ps = compute_distribution(potential_summary)

            kl = kl_div(pd, ps)
            if kl < min_kl:
                min_kl = kl
                closest_sentence = sentence

        if closest_sentence is None:
            break

        summary.append(closest_sentence)

    return summary

::: {dropdown} Code Explanation

```python
def kl_sum_verdict(reviews, summary_length=2):

    all_reviews = " ".join(reviews)
    sentences = sent_tokenize(all_reviews)
    pd = compute_distribution(all_reviews)
```
This function first combines all reviews into a single text, splits the text into individual sentences, and calculates the word distribution `pd` for the entire set of reviews.

```python
    summary = []

    while len(summary) < summary_length:
        closest_sentence = None
        min_kl = np.inf
```
An empty `summary` list is initialized. The algorithm enters a loop, selecting sentences one at a time until the desired summary length is reached. `min_kl` keeps track of the lowest KL divergence found in each iteration.

```python
        for sentence in sentences:
            if sentence in summary:
                continue

            potential_summary = " ".join(summary + [sentence])
            ps = compute_distribution(potential_summary)

            kl = kl_div(pd, ps)
            if kl < min_kl:
                min_kl = kl
                closest_sentence = sentence
```
Each unused sentence is tested by adding it to the current summary to form a `potential_summary`. Its word distribution `ps` is computed and compared to the full document distribution `pd` using KL divergence. The sentence that minimizes the divergence is selected.

```python
        if closest_sentence is None:
            break

        summary.append(closest_sentence)

    return summary
```
If no new sentence is found (which can happen if all sentences are used), the loop ends. Otherwise, the best sentence is added to the summary. The final result is a list of sentences that preserve the word distribution of the original reviews as closely as possible.
:::


The functions kl_sum_pros() and kl_sum_cons() work exactly like kl_sum_verdict(), the only difference being the list of sentences is filtered to include only positive or negative sentences.

In [81]:
def kl_sum_pros(reviews, summary_length=3):

    sentences = get_positive_sentences(reviews)
    all_reviews = " ".join(reviews)
    pd = compute_distribution(all_reviews)

    summary = []

    while len(summary) < summary_length:
        closest_sentence = None
        min_kl = np.inf

        for sentence in sentences:
            if sentence in summary:
                continue

            potential_summary = " ".join(summary + [sentence])
            ps = compute_distribution(potential_summary)

            kl = kl_div(pd, ps)
            if kl < min_kl:
                min_kl = kl
                closest_sentence = sentence

        if closest_sentence is None:
            break

        summary.append(closest_sentence)

    return summary

In [82]:
def kl_sum_cons(reviews, summary_length=3):

    sentences = get_negative_sentences(reviews)
    all_reviews = " ".join(reviews)
    pd = compute_distribution(all_reviews)

    summary = []

    while len(summary) < summary_length:
        closest_sentence = None
        min_kl = np.inf

        for sentence in sentences:
            if sentence in summary:
                continue

            potential_summary = " ".join(summary + [sentence])
            ps = compute_distribution(potential_summary)

            kl = kl_div(pd, ps)
            if kl < min_kl:
                min_kl = kl
                closest_sentence = sentence

        if closest_sentence is None:
            break

        summary.append(closest_sentence)

    return summary

In [83]:
# Generate summaries for the test set

kl_sum_verdicts = test_df['similarity_reviews'].apply(kl_sum_verdict).reset_index(drop=True)
kl_sum_pros = test_df['similarity_reviews'].apply(kl_sum_pros).reset_index(drop=True)
kl_sum_cons = test_df['similarity_reviews'].apply(kl_sum_cons).reset_index(drop=True)

In [84]:
# Combine the generated summaries into a structured format

kl_sum_generated = []

for i in range(test_df.shape[0]):
    product_summaries = {'verdict': " ".join(kl_sum_verdicts[i]), 'pros': " ".join(kl_sum_pros[i]), 'cons': " ".join(kl_sum_cons[i])}
    kl_sum_generated.append(product_summaries)

### LexRank

Since LexRank is a bit harder to implement, we will be using the [lexrank](https://pypi.org/project/lexrank/) package, which provides an easy-to-use implementation.

In [85]:
def get_lexrank(documents):
    processed_docs = []
    for doc in documents:
        if isinstance(doc, str) and len(doc.strip()) > 0:
            sents = sent_tokenize(doc)
            if len(sents) > 0:
                processed_docs.append(sents)

    lexrank = LexRank(processed_docs, stopwords=STOPWORDS['en'])
    return lexrank, processed_docs

def lexrank_verdict(documents, num_sentences=2):
    lexrank, processed_docs = get_lexrank(documents)
    sentences = [sentence for doc in processed_docs for sentence in doc]
    summary = lexrank.get_summary(sentences, summary_size=num_sentences, threshold=0.1)
    return " ".join(summary)

def lexrank_pros(documents, num_sentences=3):
    lexrank, processed_docs = get_lexrank(documents)
    sentences = get_positive_sentences(documents)
    summary = lexrank.get_summary(sentences, summary_size=num_sentences, threshold=0.1)
    return " ".join(summary)

def lexrank_cons(documents, num_sentences=3):
    lexrank, processed_docs = get_lexrank(documents)
    sentences = get_negative_sentences(documents)
    summary = lexrank.get_summary(sentences, summary_size=num_sentences, threshold=0.1)
    return " ".join(summary)

::: {dropdown} Code Explanation

```python
def get_lexrank(documents):
    processed_docs = []
    for doc in documents:
        if isinstance(doc, str) and len(doc.strip()) > 0:
            sents = sent_tokenize(doc)
            if len(sents) > 0:
                processed_docs.append(sents)

    lexrank = LexRank(processed_docs, stopwords=STOPWORDS['en'])
    return lexrank, processed_docs
```
This function prepares documents for LexRank summarization. It tokenizes each non-empty string into sentences, filters out empty or invalid entries, and builds a `LexRank` model using the processed data. It returns both the LexRank model and the processed sentence lists.

```python
def lexrank_verdict(documents, num_sentences=2):
    lexrank, processed_docs = get_lexrank(documents)
    sentences = [sentence for doc in processed_docs for sentence in doc]
    summary = lexrank.get_summary(sentences, summary_size=num_sentences, threshold=0.1)
    return " ".join(summary)
```
This function generates a LexRank-based summary by flattening all tokenized sentences and selecting the top `num_sentences` most central ones. It returns the summary as a single string.

```python
def lexrank_pros(documents, num_sentences=3):
    lexrank, processed_docs = get_lexrank(documents)
    sentences = get_positive_sentences(documents)
    summary = lexrank.get_summary(sentences, summary_size=num_sentences, threshold=0.1)
    return " ".join(summary)
```
This function filters the input documents to extract only positive sentences (e.g., pros), and applies LexRank to those. It returns a concise summary focused on positive aspects.

```python
def lexrank_cons(documents, num_sentences=3):
    lexrank, processed_docs = get_lexrank(documents)
    sentences = get_negative_sentences(documents)
    summary = lexrank.get_summary(sentences, summary_size=num_sentences, threshold=0.1)
    return " ".join(summary)
```
Similarly, this function focuses on negative sentences (e.g., cons) and summarizes them using LexRank. It returns a brief overview of negative points from the input documents.

:::

In [86]:
# Generate summaries for the test set

lexrank_verdicts = test_df['similarity_reviews'].apply(lexrank_verdict).reset_index(drop=True)
lexrank_pros = test_df['similarity_reviews'].apply(lexrank_pros).reset_index(drop=True)
lexrank_cons = test_df['similarity_reviews'].apply(lexrank_cons).reset_index(drop=True)

In [87]:
# Combine the generated summaries into a structured format

lexrank_generated = []

for i in range(test_df.shape[0]):
    product_summaries = {'verdict': lexrank_verdicts[i], 'pros': lexrank_pros[i], 'cons': lexrank_cons[i]}
    lexrank_generated.append(product_summaries)

## Results

Now that our summaries are generated, let's use ROUGE and BERTScore to evaluate how well the verdicts, pros, and cons match the gold summaries.

## ROUGE

In [98]:
# Calculate average ROUGE scores for the generated summaries

def get_rouge_scores(generated_summaries, gold_summaries):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for gen, gold in zip(generated_summaries, gold_summaries):
        scores = scorer.score(gold, gen)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)

    # Calculate averages
    avg_rouge1 = round(np.mean(rouge1_scores).item(), 4)
    avg_rouge2 = round(np.mean(rouge2_scores).item(), 4)
    avg_rougeL = round(np.mean(rougeL_scores).item(), 4)


    return {
        'rouge1': avg_rouge1,
        'rouge2': avg_rouge2,
        'rougeL': avg_rougeL
    }

In [99]:
def evaluate_rouge(generated, gold):
    # Extract generated parts
    gen_verdicts = [item['verdict'] for item in generated]
    gen_pros = [' '.join(item['pros']) for item in generated]
    gen_cons = [' '.join(item['cons']) for item in generated]

    # Extract gold parts
    gold_verdicts = [item['verdict'] for item in gold]
    gold_pros = [' '.join(item['pros']) for item in gold]
    gold_cons = [' '.join(item['cons']) for item in gold]

    # Evaluate each component
    verdict_rouge = get_rouge_scores(gen_verdicts, gold_verdicts)
    pros_rouge = get_rouge_scores(gen_pros, gold_pros)
    cons_rouge = get_rouge_scores(gen_cons, gold_cons)

    return {
        'verdict': verdict_rouge,
        'pros': pros_rouge,
        'cons': cons_rouge
    }

In [100]:
kl_sum_rouge_results = evaluate_rouge(kl_sum_generated, gold_summaries)

print("KL-Sum ROUGE Results:\n")
print(f"VERDICT:{kl_sum_rouge_results['verdict']}")
print(f"PROS:{kl_sum_rouge_results['pros']}")
print(f"CONS:{kl_sum_rouge_results['cons']}")

KL-Sum ROUGE Results:

VERDICT:{'rouge1': 0.1404, 'rouge2': 0.0102, 'rougeL': 0.0917}
PROS:{'rouge1': 0.007, 'rouge2': 0.0, 'rougeL': 0.007}
CONS:{'rouge1': 0.0076, 'rouge2': 0.0, 'rougeL': 0.0076}


In [101]:
lexrank_rouge_results = evaluate_rouge(lexrank_generated, gold_summaries)

print("LexRank ROUGE Results:\n")
print(f"VERDICT: {lexrank_rouge_results['verdict']}")
print(f"PROS: {lexrank_rouge_results['pros']}")
print(f"CONS: {lexrank_rouge_results['cons']}")

LexRank ROUGE Results:

VERDICT: {'rouge1': 0.1576, 'rouge2': 0.0136, 'rougeL': 0.1134}
PROS: {'rouge1': 0.0143, 'rouge2': 0.0, 'rougeL': 0.0142}
CONS: {'rouge1': 0.0083, 'rouge2': 0.0, 'rougeL': 0.0083}


Both algorithms generate similar ROUGE scores, with LexRank slightly outperforming KL-Sum. These scores can be interpreted as percentages of matchings between generated and gold summaries. On average, about 14% and 16% of unigrams (single words) in the gold verdicts were matched by the generated ones, while bigrams rarely matched. Pros and cons were much less accurate, which is to be expected from extractive models that are not designed for this type of summarization.

## BERTScore



In [102]:
# Calculate average BERTScore for the generated summaries

def get_bertscore_scores(generated_summaries, gold_summaries, lang='en'):
    P, R, F1 = bert_score(generated_summaries, gold_summaries, lang=lang, verbose=False)
    return {
        'precision': round(P.numpy().mean().item(), 4),
        'recall': round(R.numpy().mean().item(), 4),
        'f1': round(F1.numpy().mean().item(), 4)
    }

In [103]:
# Apply BERTScore evaluation to each summary component

def evaluate_bertscore(generated, gold, lang='en'):
    # Extract generated parts
    gen_verdicts = [item['verdict'] for item in generated]
    gen_pros = [' '.join(item['pros']) for item in generated]
    gen_cons = [' '.join(item['cons']) for item in generated]

    # Extract gold/reference parts
    gold_verdicts = [item['verdict'] for item in gold]
    gold_pros = [' '.join(item['pros']) for item in gold]
    gold_cons = [' '.join(item['cons']) for item in gold]

    # Evaluate BERTScore for each component
    verdict_scores = get_bertscore_scores(gen_verdicts, gold_verdicts, lang)
    pros_scores = get_bertscore_scores(gen_pros, gold_pros, lang)
    cons_scores = get_bertscore_scores(gen_cons, gold_cons, lang)

    return {
        'verdict': verdict_scores,
        'pros': pros_scores,
        'cons': cons_scores
    }

In [104]:
kl_sum_bertscore_results = evaluate_bertscore(kl_sum_generated, gold_summaries)

print("KL-Sum BERTScore Results:\n")
print(f"VERDICT:{kl_sum_bertscore_results['verdict']}")
print(f"PROS:{kl_sum_bertscore_results['pros']}")
print(f"CONS:{kl_sum_bertscore_results['cons']}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


KL-Sum BERTScore Results:

VERDICT:{'precision': 0.8258, 'recall': 0.8595, 'f1': 0.8423}
PROS:{'precision': 0.5786, 'recall': 0.7773, 'f1': 0.6633}
CONS:{'precision': 0.5725, 'recall': 0.7849, 'f1': 0.6619}


In [105]:
lexrank_bertscore_results = evaluate_bertscore(lexrank_generated, gold_summaries)

print("LexRank BERTScore Results:\n")
print(f"VERDICT: {lexrank_bertscore_results['verdict']}")
print(f"PROS: {lexrank_bertscore_results['pros']}")
print(f"CONS: {lexrank_bertscore_results['cons']}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LexRank BERTScore Results:

VERDICT: {'precision': 0.8612, 'recall': 0.8594, 'f1': 0.8602}
PROS: {'precision': 0.6063, 'recall': 0.7809, 'f1': 0.6824}
CONS: {'precision': 0.578, 'recall': 0.7853, 'f1': 0.6657}


Similar to our ROUGE scores, these models generated much more accurate summaries for verdicts than for pros and cons. Still, the BERTScores indicate that the generated pros and cons are reasonably informative. High recall scores indicate that most of the important content from the gold pros list is contained in the generated summaries.

In [107]:
# Save results

results_to_save = {
    "kl_sum_rouge_results": kl_sum_rouge_results,
    "lexrank_rouge_results": lexrank_rouge_results,
    "kl_sum_bertscore_results": kl_sum_bertscore_results,
    "lexrank_bertscore_results": lexrank_bertscore_results,
    "kl_sum_generated": kl_sum_generated,
    "lexrank_generated": lexrank_generated
}

# Save each dictionary as a JSON file
for name, data in results_to_save.items():
    file_path = f"../data/{name}.json"
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

print("All results saved.")

All results saved.
